# 02 — Traffic Forecasting Models

Trains and evaluates 4 models (Persistence, Seasonal, Random Forest, XGBoost)
for 15, 30, and 60-minute speed forecasting.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, str(Path('.').resolve().parent))
from config.settings import CLEANED_DATA_PATH, MODELS_DIR, REPORTS_DIR, FORECAST_HORIZONS_MIN
from src.feature_engineering import make_features_for_sensor, chronological_split, get_X_y, FEATURE_COLS
from src.train_model import train_all

plt.style.use('dark_background')
print('Libraries loaded.')

In [ ]:
# Load data
df = pd.read_csv(CLEANED_DATA_PATH, parse_dates=['timestamp'])
df['sensor_id'] = df['sensor_id'].astype(str)
print(f'Loaded: {len(df):,} rows, {df.sensor_id.nunique()} sensors')

# Pick one sensor for demonstration
demo_sensor = df.groupby('sensor_id')['observed_speed'].count().idxmax()
print(f'Demo sensor (most complete): {demo_sensor}')

In [ ]:
# Feature engineering demo for one sensor, one horizon
feat_df = make_features_for_sensor(df, demo_sensor, horizon_min=15)
print(f'Feature matrix shape: {feat_df.shape}')
print(f'Feature columns ({len(FEATURE_COLS)}):')
for i, c in enumerate(FEATURE_COLS):
    print(f'  {i+1:2d}. {c}')
print(f'\nTarget: target_speed (speed 15 min ahead)')
feat_df.head(3)

In [ ]:
# Chronological split
train, val, test = chronological_split(feat_df)
print(f'Train: {len(train):,} rows ({train.timestamp.min().date()} → {train.timestamp.max().date()})')
print(f'Val:   {len(val):,} rows ({val.timestamp.min().date()} → {val.timestamp.max().date()})')
print(f'Test:  {len(test):,} rows ({test.timestamp.min().date()} → {test.timestamp.max().date()})')

In [ ]:
# Run full training pipeline
print('Training all models on top 10 sensors...')
results = train_all()
print('\nTraining complete!')

In [ ]:
# Load and display metrics
with open(REPORTS_DIR / 'model_metrics.json') as f:
    metrics = json.load(f)

rows = []
for h, hr in metrics['horizons'].items():
    for model, m in hr['models'].items():
        rows.append({'Horizon': f'{h} min', 'Model': model.replace('_',' ').title(),
                     'Test MAE': round(m.get('test_mae',np.nan),3),
                     'Test RMSE': round(m.get('test_rmse',np.nan),3),
                     'Best': '*' if hr['best_model']==model else ''})

res_df = pd.DataFrame(rows)
print('=== Model Evaluation Results ===')
print(res_df.to_string(index=False))

In [ ]:
# MAE comparison chart
horizons = sorted(set(res_df['Horizon'].tolist()))
models = [m for m in res_df['Model'].unique() if 'nan' not in str(res_df[res_df['Model']==m]['Test MAE'].values[0])]
colors = ['#00D4FF', '#00C851', '#FFD700', '#FF8C00']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, metric in enumerate(['Test MAE', 'Test RMSE']):
    pivot = res_df.pivot(index='Horizon', columns='Model', values=metric)
    pivot.plot(kind='bar', ax=axes[ax_idx], color=colors[:len(pivot.columns)])
    axes[ax_idx].set_title(f'{metric} by Model & Horizon', fontsize=13)
    axes[ax_idx].set_xlabel('Forecast Horizon')
    axes[ax_idx].set_ylabel(f'{metric} (mph)')
    axes[ax_idx].tick_params(axis='x', rotation=0)
    axes[ax_idx].legend(fontsize=8)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'figures' / '07_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')